# Deterministic RF/channel augmentation

This notebook demonstrates the generation-time RF/channel pipeline added in MR3. The pipeline is intentionally separate from file readers: augment a generated `Signal` before writing it, then readers return the stored samples unchanged.

The fixed transform order is: optional normalization before, frequency offset, phase noise, Rayleigh fading, AWGN, and optional normalization after. Every random choice is derived from the global seed, stable record identity, and effect name.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from torchsig.signals.signal_types import Signal
from torchsig.transforms.rf_channel import (
    FadingConfig,
    RFChannelImpairmentConfig,
    RFChannelImpairmentPipeline,
    UniformRange,
)

SAMPLE_RATE_HZ = 1_000_000.0
NUM_SAMPLES = 4096
time_s = np.arange(NUM_SAMPLES) / SAMPLE_RATE_HZ
clean_iq = np.exp(2j * np.pi * 100_000.0 * time_s).astype(np.complex64)

## Declare the generation policy

Ranges are sampled uniformly. A scalar can be represented by equal minimum and maximum bounds. Frequency offset is in Hz, phase noise is a Gaussian standard deviation in degrees, fading coherence bandwidth is normalized to sample rate, and AWGN power is absolute dB. Configuration is validated when these objects are created.

In [ ]:
config = RFChannelImpairmentConfig(
    frequency_offset_hz=UniformRange(-30_000.0, 30_000.0),
    phase_noise_degrees=UniformRange(0.5, 2.0),
    fading=FadingConfig(
        coherence_bandwidth=UniformRange(0.03, 0.08),
        power_delay_profile=(1.0, 0.5, 0.1),
    ),
    noise_power_db=UniformRange(-45.0, -35.0),
    normalization="after",
)
pipeline = RFChannelImpairmentPipeline(config=config, global_seed=2026)
config

The same policy can be loaded from a JSON/YAML-style mapping with `RFChannelImpairmentConfig.from_dict(...)`, which is convenient for dataset generator configuration files.

In [ ]:
mapping_config = RFChannelImpairmentConfig.from_dict({
    "frequency_offset_hz": [-30_000.0, 30_000.0],
    "phase_noise_degrees": [0.5, 2.0],
    "fading": {
        "coherence_bandwidth": [0.03, 0.08],
        "power_delay_profile": [1.0, 0.5, 0.1],
    },
    "noise_power_db": [-45.0, -35.0],
    "normalization": "after",
})
assert mapping_config == config

## Augment one generated record

Use a stable record identity from the generator's record specification, not a worker number or completion index. `apply` mutates and returns the supplied `Signal`.

In [ ]:
record = Signal(data=clean_iq.copy(), class_name="example_tone")
augmented = pipeline.apply(
    record,
    record_identity="train/example_tone/000042",
    sample_rate=SAMPLE_RATE_HZ,
)
provenance = augmented["rf_channel_impairments"]
provenance

The provenance contains the actual sampled values and applied order. This metadata should be written alongside the IQ record.

In [ ]:
assert provenance["order"] == [
    "frequency_offset",
    "phase_noise",
    "fading",
    "awgn",
    "normalize",
]
print("Mean output power:", np.mean(np.abs(augmented.data) ** 2))
for effect in provenance["applied"]:
    print(effect)

## Compare clean and augmented samples

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
view = slice(0, 300)
axes[0, 0].plot(clean_iq.real[view], label="clean I")
axes[0, 0].plot(augmented.data.real[view], label="augmented I", alpha=0.8)
axes[0, 0].set_title("Time-domain I component")
axes[0, 0].legend()
axes[0, 1].scatter(clean_iq.real[::8], clean_iq.imag[::8], s=5, label="clean")
axes[0, 1].scatter(augmented.data.real[::8], augmented.data.imag[::8], s=5, alpha=0.5, label="augmented")
axes[0, 1].set_title("IQ plane")
axes[0, 1].axis("equal")
axes[0, 1].legend()
frequency_hz = np.fft.fftshift(np.fft.fftfreq(NUM_SAMPLES, 1 / SAMPLE_RATE_HZ))
clean_spectrum = 20 * np.log10(np.maximum(np.abs(np.fft.fftshift(np.fft.fft(clean_iq))), 1e-12))
augmented_spectrum = 20 * np.log10(np.maximum(np.abs(np.fft.fftshift(np.fft.fft(augmented.data))), 1e-12))
axes[1, 0].plot(frequency_hz / 1e3, clean_spectrum, label="clean")
axes[1, 0].plot(frequency_hz / 1e3, augmented_spectrum, label="augmented", alpha=0.8)
axes[1, 0].set_title("Spectrum")
axes[1, 0].set_xlabel("Frequency (kHz)")
axes[1, 0].set_ylabel("Magnitude (dB)")
axes[1, 0].legend()
axes[1, 1].axis("off")
axes[1, 1].text(0, 1, "Applied order:\n" + "\n".join(provenance["order"]), va="top", family="monospace")
fig.tight_layout()

## Reproducibility across execution order

A record's result depends on its identity, not the order in which workers process records. Here the same identities are processed in opposite orders and compared.

In [ ]:
def generate(identity):
    signal = Signal(data=clean_iq.copy())
    return pipeline.apply(signal, record_identity=identity, sample_rate=SAMPLE_RATE_HZ)

identities = ["record-0", "record-1", "record-2"]
forward = {identity: generate(identity) for identity in identities}
reverse = {identity: generate(identity) for identity in reversed(identities)}
for identity in identities:
    np.testing.assert_array_equal(forward[identity].data, reverse[identity].data)
    assert forward[identity]["rf_channel_impairments"] == reverse[identity]["rf_channel_impairments"]
print("All records reproduced exactly across execution orders.")

Different identities produce independent draws, while an effect's draw remains stable if other effects are enabled or disabled because each effect has its own derived seed.

In [ ]:
for identity in identities:
    sampled = forward[identity]["rf_channel_impairments"]["applied"]
    offset = next(item["offset_hz"] for item in sampled if item["name"] == "frequency_offset")
    print(identity, f"frequency offset = {offset:.1f} Hz")

## Demonstrate each effect independently

In [ ]:
independent_configs = {
    "frequency offset": RFChannelImpairmentConfig(frequency_offset_hz=UniformRange(25_000, 25_000)),
    "phase noise": RFChannelImpairmentConfig(phase_noise_degrees=UniformRange(2, 2)),
    "fading": RFChannelImpairmentConfig(fading=FadingConfig(UniformRange(0.05, 0.05))),
    "AWGN": RFChannelImpairmentConfig(noise_power_db=UniformRange(-35, -35)),
}

for name, effect_config in independent_configs.items():
    effect_pipeline = RFChannelImpairmentPipeline(effect_config, global_seed=2026)
    result = effect_pipeline.apply(Signal(data=clean_iq.copy()), record_identity=0, sample_rate=SAMPLE_RATE_HZ)
    print(name, "->", result["rf_channel_impairments"]["applied"])

## Disabled configuration is a true no-op

The default configuration returns the same `Signal` object without changing its samples or adding provenance metadata. This preserves existing generator behavior unless augmentation is explicitly enabled.

In [ ]:
disabled = RFChannelImpairmentPipeline()
unchanged = Signal(data=clean_iq.copy(), class_name="example_tone")
result = disabled.apply(unchanged, record_identity=0, sample_rate=SAMPLE_RATE_HZ)
assert result is unchanged
np.testing.assert_array_equal(result.data, clean_iq)
assert "rf_channel_impairments" not in result.keys()
print("Disabled pipeline preserved samples and metadata.")